<a href="https://colab.research.google.com/github/jotaeleb/tif-ciencias-de-datos/blob/main/notebooks/proyecto_modelado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️TIF Análisis de Detección de Intrusiones en Red

**Grupo 1**: Gonza Gabriela · Casasola Hernán · Biazutti Luciano · Lera Aníbal Iván · Alvarado Marcelo

**Módulo**: Ciencias de Datos y Optimización de Modelos

**Carrera**: Tecnicatura Universitaria en Ciencias de Datos e IA Aplicada — UPATECO

---

# 1. Librerías

In [58]:
import io
import requests

import numpy as np
import pandas as pd

from datetime import datetime

from pandas.api.types import is_string_dtype

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, f1_score, recall_score, accuracy_score, classification_report

from imblearn.over_sampling import SMOTE

from IPython.display import display, HTML



# 2. Funciones auxiliares


In [54]:
#Esta función carga un dataset desde el repo de Github
def cargarDataset(archivos):

  GITHUB_USER = 'jotaeleb'
  GITHUB_REPO = 'tif-ciencias-de-datos'
  GITHUB_BRANCH = 'main'
  DATASET_DIR = 'dataset'

  BASE_URL = (
    f'https://raw.githubusercontent.com/'
    f'{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{DATASET_DIR}'
  )

  partes = []
  for archivo in archivos:
    url = f'{BASE_URL}/{archivo}'
    print(f"Cargando archivo {url}")
    partes.append(pd.read_parquet(io.BytesIO(requests.get(url).content)))


  completo = pd.concat(partes,ignore_index=True)
  completo['Label'] = completo['Label'].str.replace(r'[^\w\s-]', '-', regex=True)

  return completo

#Esta función devuelve las columnas numéricas del dataset
def colNumericas(dFrame):
  columnas = []
  for columna, tipoDato in dFrame.dtypes.items():
    if not is_string_dtype(tipoDato):
      columnas.append(columna)
  return columnas

#Esta función evalúa el modelo
def evaluarModelo(y_prueba, predicciones,clases=None):
  accuracy = accuracy_score(y_prueba, predicciones)
  f1Macro = f1_score(y_prueba, predicciones, average='macro')
  print(f"Accuracy: {accuracy:.3f} | F1 Macro: {f1Macro:.3f}")

  if (clases is not None):
    reporte = classification_report(y_prueba,predicciones,target_names=clases,output_dict=True)
  else:
    reporte = classification_report(y_prueba,predicciones,output_dict=True)


  dfReporte = pd.DataFrame(reporte).transpose()

  dfReporte = dfReporte.round(4)

  display(HTML(dfReporte.to_html()))

  #Esta función lista categorias codificadas
  def verCategorias():
    pass



##  Carga Dataset limpio con submuestreo
---



In [8]:
print("--> Carga Dataset <--")
df = cargarDataset(["dataset_limpio_submuestreo.parquet"])
print(df.shape)

--> Carga Dataset <--
Cargando archivo https://raw.githubusercontent.com/jotaeleb/tif-ciencias-de-datos/main/dataset/dataset_limpio_submuestreo.parquet
(168004, 66)


##  Categorías
---






In [9]:
df['Label'].value_counts()

,count
Label,
BENIGN,50000
PortScan,50000
DoS-DDoS,50000
Brute Force,13826
Web Attack,2159
Botnet,1956
Rare Attack,63


## Pre requisitos comunes para los modelos Baseline
---



In [23]:
#Obtener listado de las columnas numéricas del dataset
columnas = colNumericas(df)

#Asignar a X las columnas numéricas del dataset
X = df[columnas]

#Asignar a etiqueta (Y) la variable objetivo
etiqueta = df["Label"]



##  Modelo Baseline Dummy
---


In [55]:
X_treino, X_prueba, y_treino, y_prueba = train_test_split(X, etiqueta, test_size=0.2, random_state=42, stratify=etiqueta)

modeloBaselineDummy = DummyClassifier(strategy='stratified')

modeloBaselineDummy.fit(X_treino, y_treino)

predicciones = modeloBaselineDummy.predict(X_prueba)

evaluarModelo(y_prueba, predicciones)


Accuracy: 0.268 | F1 Macro: 0.140


,precision,recall,f1-score,support
BENIGN,0.2912,0.2947,0.2930,10000.000
Botnet,0.0163,0.0153,0.0158,391.000
Brute Force,0.0800,0.0788,0.0794,2765.000
DoS-DDoS,0.2902,0.2892,0.2897,10000.000
PortScan,0.2949,0.2937,0.2943,10000.000
Rare Attack,0.0000,0.0000,0.0000,13.000
Web Attack,0.0089,0.0093,0.0091,432.000
accuracy,0.2680,0.2680,0.2680,0.268
macro avg,0.1402,0.1401,0.1402,33601.000
weighted avg,0.2677,0.2680,0.2678,33601.000


## Modelo Baseline Regresión Logística
---


In [35]:
X_treino, X_prueba, y_treino, y_prueba = train_test_split(X, etiqueta, test_size=0.2, random_state=42, stratify=etiqueta)

modeloBaselineRegresionLogistica = make_pipeline(
  StandardScaler(),
  LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
)

modeloBaselineRegresionLogistica.fit(X_treino, y_treino)

predicciones = modeloBaselineRegresionLogistica.predict(X_prueba)
evaluarModelo(y_prueba, predicciones)


Accuracy: 0.951 | F1 Macro: 0.762


,precision,recall,f1-score,support
BENIGN,0.9963,0.8521,0.9186,10000.0000
Botnet,0.3859,0.9949,0.5561,391.0000
Brute Force,0.9403,0.9967,0.9677,2765.0000
DoS-DDoS,0.9678,0.9892,0.9784,10000.0000
PortScan,0.9852,0.9984,0.9918,10000.0000
Rare Attack,0.0282,0.5385,0.0536,13.0000
Web Attack,0.8063,0.9444,0.8699,432.0000
accuracy,0.9511,0.9511,0.9511,0.9511
macro avg,0.7300,0.9020,0.7623,33601.0000
weighted avg,0.9700,0.9511,0.9570,33601.0000


## Modelo Baseline XGBoost
---


In [56]:
label_encoder = LabelEncoder()

y_XGBoost = label_encoder.fit_transform(etiqueta)

X_treino, X_prueba, y_treino, y_prueba = train_test_split(X, y_XGBoost, test_size=0.2, random_state=42, stratify=etiqueta)

modeloBaselineXGBoost = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8
    )

modeloBaselineXGBoost.fit(X_treino, y_treino)
predicciones = modeloBaselineXGBoost.predict(X_prueba)
evaluarModelo(y_prueba, predicciones,label_encoder.classes_)


Accuracy: 0.999 | F1 Macro: 0.965


,precision,recall,f1-score,support
BENIGN,0.9981,0.9988,0.9985,10000.0000
Botnet,0.9872,0.9847,0.9859,391.0000
Brute Force,1.0000,1.0000,1.0000,2765.0000
DoS-DDoS,0.9995,0.9995,0.9995,10000.0000
PortScan,0.9995,0.9995,0.9995,10000.0000
Rare Attack,0.9000,0.6923,0.7826,13.0000
Web Attack,0.9953,0.9884,0.9919,432.0000
accuracy,0.9989,0.9989,0.9989,0.9989
macro avg,0.9828,0.9519,0.9654,33601.0000
weighted avg,0.9989,0.9989,0.9989,33601.0000


## Modelo Baseline XGBoost + SMOTE
---


In [61]:
label_encoder = LabelEncoder()

y_XGBoost = label_encoder.fit_transform(etiqueta)

mapeoEtiquetas = dict(zip(label_encoder.classes_,range(len(label_encoder.classes_))))

print(mapeoEtiquetas)

X_treino, X_prueba, y_treino, y_prueba = train_test_split(X, y_XGBoost, test_size=0.2, random_state=42, stratify=etiqueta)

smote = SMOTE(
        sampling_strategy={
            mapeoEtiquetas['Brute Force']: 20000,
            mapeoEtiquetas['Web Attack']: 12000,
            mapeoEtiquetas['Botnet']: 12000,
            mapeoEtiquetas['Rare Attack']: 4000
        },
        k_neighbors=3,  # reducir k porque Rare Attack tiene solo 63
        random_state=42)

X_balanceado, y_balanceado = smote.fit_resample(X_treino, y_treino)

print("Clases balanceadas con SMOTE")
valores, frecuencias = np.unique(y_balanceado, return_counts=True)

for clase, freq in zip(valores, frecuencias):
  print(f"{label_encoder.inverse_transform([clase])[0]}: {freq}")

modeloBaselineXGBoost = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8
    )

modeloBaselineXGBoost.fit(X_balanceado, y_balanceado)
predicciones = modeloBaselineXGBoost.predict(X_prueba)
evaluarModelo(y_prueba, predicciones,label_encoder.classes_)

{'BENIGN': 0, 'Botnet': 1, 'Brute Force': 2, 'DoS-DDoS': 3, 'PortScan': 4, 'Rare Attack': 5, 'Web Attack': 6}
Clases balanceadas con SMOTE
BENIGN: 40000
Botnet: 12000
Brute Force: 20000
DoS-DDoS: 40000
PortScan: 40000
Rare Attack: 4000
Web Attack: 12000
Accuracy: 0.999 | F1 Macro: 0.945


,precision,recall,f1-score,support
BENIGN,0.9988,0.9981,0.9984,10000.0000
Botnet,0.9748,0.9898,0.9822,391.0000
Brute Force,0.9996,1.0000,0.9998,2765.0000
DoS-DDoS,0.9995,0.9993,0.9994,10000.0000
PortScan,0.9995,0.9996,0.9996,10000.0000
Rare Attack,0.5556,0.7692,0.6452,13.0000
Web Attack,0.9953,0.9861,0.9907,432.0000
accuracy,0.9987,0.9987,0.9987,0.9987
macro avg,0.9319,0.9632,0.9450,33601.0000
weighted avg,0.9988,0.9987,0.9987,33601.0000
